# HSR IK Grasp on Google Colab (headless rendering)

This notebook runs the `IK_grasp_hsr` example from the [hsr-genesis](https://github.com/icsl-aist/hsr-genesis) repository on Google Colab and renders the resulting animation inline using `mediapy`.

The headless-rendering technique (EGL ICD config + `show_viewer=False` + offscreen `scene.add_camera` + `cam.render()` per step + `media.show_video`) follows the approach in [Taka-Hashimoto's hello_genesis gist](https://gist.github.com/Taka-Hashimoto/9546241b78e6d2c85a270c39a41808ff).

> **Note:** Run this on a GPU runtime (Colab: *Runtime → Change runtime type → T4 GPU* or better).

## 1. Install dependencies

In [ ]:
# Genesis (pinned to the version hsr-genesis depends on) + mediapy for inline video.
# Pin setuptools<82 for compatibility with the torch wheel Colab ships, and
# install jedi to satisfy IPython's dependency requirement.
!pip install 'setuptools<82' jedi -q
!pip install genesis-world==0.4.6 -q
!pip install mediapy -q

## 2. Clone & install the hsr-genesis repo

We need the `hsr_genesis` package (HSR-specific entity, controllers, analytic IK) and the bundled `data/urdf/hsrb4s.urdf`.

In [ ]:
import os, subprocess, pathlib, sys

REPO_DIR = pathlib.Path('/content/hsr-genesis')
if not REPO_DIR.exists():
    # Clone with --recurse-submodules so data/urdf/hsrb_meshes (and
    # data/tmc_wrs_gazebo) are fetched too — the URDF needs these meshes.
    subprocess.run(['git', 'clone', '--depth', '1', '--recurse-submodules',
                    '--shallow-submodules',
                    'https://github.com/icsl-aist/hsr-genesis.git',
                    str(REPO_DIR)], check=True)
else:
    # If the repo was cloned without submodules, fetch them now.
    subprocess.run(['git', '-C', str(REPO_DIR), 'submodule', 'update',
                    '--init', '--recursive', '--depth', '1'], check=True)

# Editable install so `hsr_genesis` is importable and data paths resolve.
# (not -q so any install errors are visible)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(REPO_DIR)],
               check=True)

# Fallback: also add src/ to PYTHONPATH in case the editable install's
# .pth shim wasn't picked up by the current kernel (per AGENTS.md).
SRC_DIR = str(REPO_DIR / 'src')
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

REPO_ROOT = REPO_DIR
URDF_PATH = REPO_ROOT / 'data' / 'urdf' / 'hsrb4s.urdf'
MESHES_DIR = REPO_ROOT / 'data' / 'urdf' / 'hsrb_meshes'
print('Repo root :', REPO_ROOT)
print('URDF path :', URDF_PATH, '| exists =', URDF_PATH.exists())
print('Meshes dir:', MESHES_DIR, '| exists =', MESHES_DIR.exists(),
      '| non-empty =', MESHES_DIR.exists() and any(MESHES_DIR.iterdir()))

# Verify the import actually works before we proceed.
import hsr_genesis
print('hsr_genesis imported from:', pathlib.Path(hsr_genesis.__file__).parent)

## 3. Configure NVIDIA EGL ICD (headless GPU rendering on Colab)

Colab's NVIDIA driver needs an EGL vendor ICD entry so Genesis can create an offscreen EGL context without a display.

In [ ]:
NVIDIA_ICD_CONFIG_PATH = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
ICD_CONFIG_CONTENT = """{
    "file_format_version" : "1.0.0",
    "ICD" : {
        "library_path" : "libEGL_nvidia.so.0"
    }
}
"""
os.makedirs(os.path.dirname(NVIDIA_ICD_CONFIG_PATH), exist_ok=True)
with open(NVIDIA_ICD_CONFIG_PATH, 'w') as f:
    f.write(ICD_CONFIG_CONTENT)
print('Wrote EGL ICD config to', NVIDIA_ICD_CONFIG_PATH)

## 4. Initialize Genesis (GPU backend)

In [ ]:
import math
from pathlib import Path

import numpy as np
import torch
import mediapy as media
from tqdm.notebook import tqdm

import genesis as gs

# Guard against re-running this cell without a kernel restart: gs.init()
# raises if Genesis is already initialized.
if not getattr(gs, '_initialized', False):
    gs.init(backend=gs.gpu)
else:
    print('Genesis already initialized, skipping gs.init().')

## 5. Build the scene (headless + offscreen camera)

Mirrors `examples/tutorials/IK_grasp_hsr.py`, but with `show_viewer=False` and an added offscreen camera whose frames we'll capture each step.

In [ ]:
from hsr_genesis.hsr_rigid_entity import HSRBURDF, JointTrajectory
from hsr_genesis.base_controller import Trajectory
from hsr_genesis.analytic_ik import JOINT_ORDER


def _quat_wxyz_to_yaw(quat):
    quat_val = quat.detach().cpu().numpy() if isinstance(quat, torch.Tensor) else np.asarray(quat, dtype=np.float64)
    w, x, y, z = quat_val[:4]
    siny_cosp = 2.0 * (w * z + x * y)
    cosy_cosp = 1.0 - 2.0 * (y * y + z * z)
    return math.atan2(siny_cosp, cosy_cosp)


def _arm_dofs_idx_local(entity):
    dofs = []
    for name in JOINT_ORDER:
        joint_dofs = entity.get_joint(name).dofs_idx_local
        if isinstance(joint_dofs, (list, tuple)):
            dofs.extend(int(idx) for idx in joint_dofs)
        else:
            dofs.append(int(joint_dofs))
    return dofs


def _qpos_to_arm_dofs(entity, qpos, arm_dofs_idx_local):
    saved_qpos = entity.get_qpos().clone()
    try:
        entity.set_qpos(qpos, zero_velocity=False)
        dofs = entity.get_dofs_position()
    finally:
        entity.set_qpos(saved_qpos, zero_velocity=False)
    if dofs.ndim == 1:
        dofs = dofs.unsqueeze(0)
    return dofs[:, arm_dofs_idx_local]


def arm_traj_names():
    return list(JOINT_ORDER)


scene = gs.Scene(
    viewer_options=gs.options.ViewerOptions(
        camera_pos=(3, -1, 1.5),
        camera_lookat=(0.0, 0.0, 0.5),
        camera_fov=30,
        max_FPS=60,
    ),
    sim_options=gs.options.SimOptions(dt=0.02, substeps=20),
    rigid_options=gs.options.RigidOptions(use_gjk_collision=True),
    show_viewer=False,   # headless: no GUI window
    show_FPS=False,
)

scene.add_entity(gs.morphs.Plane(), visualize_contact=True)

cube_pos = np.array([0.45, 0.0, 0.02], dtype=np.float32)
cube = scene.add_entity(
    gs.morphs.Box(size=(0.04, 0.04, 0.04), pos=tuple(cube_pos.tolist())),
    visualize_contact=True,
)

hsr = scene.add_entity(
    HSRBURDF(
        file=str(URDF_PATH),
        fixed=False,
        recompute_inertia=False,
        links_to_keep=['hand_palm_link'],
        robot='hsrb',
        base_mode='planar',
        end_effector_frame='hand_palm_link',
        use_base_controller=True,
        base_control_mode='controller',
        optimizer='gpu',
    ),
    visualize_contact=True,
)

# Offscreen camera for headless rendering (cf. hello_genesis.ipynb technique).
cam = scene.add_camera(
    res=(640, 480),
    pos=(3.0, -1.0, 1.5),
    lookat=(0.0, 0.0, 0.5),
    fov=30,
    GUI=False,
)

# FT sensor with debug visualization (auto-discovers downstream links at build()).
ft_link = hsr.get_link('wrist_ft_sensor_frame')
ft_sensor = scene.add_sensor(
    gs.sensors.ForceTorque(
        entity_idx=int(hsr.idx),
        link_idx_local=int(ft_link.idx_local),
        draw_debug=True,
        debug_force_scale=0.02,
        debug_force_color=(1.0, 0.0, 0.0, 0.9),
        debug_torque_scale=0.005,
        debug_torque_color=(0.0, 0.3, 1.0, 0.9),
    )
)

scene.build()
print('Scene built. HSR idx =', int(hsr.idx))

## 6. Solve IK for the grasp pose

In [ ]:
end_effector = hsr.get_link('hand_palm_link')
hsr.end_effector_offset = [0.0, 0.0, 0.09]
hand_quat = np.array([0.0, 1.0, 0.0, 0.0], dtype=np.float32)

qpos = hsr.inverse_kinematics(
    link=end_effector,
    pos=cube_pos,
    quat=hand_quat,
    max_samples=200,
    max_solver_iters=150,
    max_step_size=0.7,
    respect_joint_limit=False,
)

arm_dofs_idx_local = _arm_dofs_idx_local(hsr)
arm_dofs = _qpos_to_arm_dofs(hsr, qpos, arm_dofs_idx_local)

target_pos = (float(qpos[0]), float(qpos[1]), float(qpos[2]))
target_yaw = _quat_wxyz_to_yaw(qpos[3:7])
print('IK target pos:', target_pos, 'yaw:', target_yaw)

## 7. Run the grasp sequence and capture frames

Three phases (approach → close gripper → lift), rendering one camera frame per sim step into `frames`.

In [ ]:
dt = float(scene.sim_options.dt)
duration = 3.0

base_traj = Trajectory(
    positions=torch.tensor([[target_pos[0], target_pos[1], target_yaw]], device=gs.device, dtype=gs.tc_float),
    time_from_start=torch.tensor([duration], device=gs.device, dtype=gs.tc_float),
)
arm_traj = JointTrajectory(
    positions=arm_dofs,
    time_from_start=torch.tensor([duration], device=gs.device, dtype=gs.tc_float),
    joint_names=arm_traj_names(),
)

hsr.set_whole_body_trajectory_batched(
    arm_trajectory=arm_traj,
    base_trajectory=base_traj,
    envs_idx=[0],
    start_time=None,
)

motor_dofs = hsr.get_joint('hand_motor_joint').dofs_idx_local
motor_idx = int(motor_dofs[0]) if isinstance(motor_dofs, (list, tuple)) else int(motor_dofs)
hand_open = torch.tensor([[1.0]], device=gs.device, dtype=gs.tc_float)

frames = []

# --- Phase 1: approach to pre-grasp pose with hand open ---
max_steps = int(duration / dt) + 50
for step in tqdm(range(max_steps), desc='approach'):
    hsr.step_whole_body_trajectory_batched(dt, envs_idx=[0])
    if step == 0:
        hsr.control_dofs_position(hand_open, dofs_idx_local=[motor_idx])
    scene.step()
    frames.append(cam.render()[0])

# --- Phase 2: close gripper (torque-controlled grasp) ---
gripper = hsr.get_gripper_batched()
effort = torch.tensor([3.0], device=gs.device, dtype=gs.tc_float)
active = torch.tensor([True], device=gs.device, dtype=torch.bool)
gripper.set_apply_force_goal(effort=effort, active_mask=active, envs_idx=[0])

for _ in tqdm(range(300), desc='grasp'):
    gripper.step_apply_force(dt, envs_idx=[0])
    hsr.step_whole_body_trajectory_batched(dt, envs_idx=[0])
    scene.step()
    frames.append(cam.render()[0])

# --- Phase 3: lift the grasped object ---
current_qpos = hsr.get_qpos().clone()
lift_height = 0.25
lift_pos = np.array([cube_pos[0], cube_pos[1], lift_height], dtype=np.float32)

lift_qpos = hsr.inverse_kinematics(
    link=end_effector,
    pos=lift_pos,
    quat=hand_quat,
    init_qpos=current_qpos,
    max_samples=200,
    max_solver_iters=150,
    max_step_size=0.7,
    respect_joint_limit=False,
)
lift_arm_dofs = _qpos_to_arm_dofs(hsr, lift_qpos, arm_dofs_idx_local)

lift_duration = 2.0
lift_arm_traj = JointTrajectory(
    positions=lift_arm_dofs,
    time_from_start=torch.tensor([lift_duration], device=gs.device, dtype=gs.tc_float),
    joint_names=arm_traj_names(),
)
hsr.set_whole_body_trajectory_batched(
    arm_trajectory=lift_arm_traj,
    base_trajectory=None,
    envs_idx=[0],
    start_time=None,
)

lift_steps = int(lift_duration / dt) + 50
for _ in tqdm(range(lift_steps), desc='lift'):
    gripper.step_apply_force(dt, envs_idx=[0])
    hsr.step_whole_body_trajectory_batched(dt, envs_idx=[0])
    scene.step()
    frames.append(cam.render()[0])

# hold a moment at the end
for _ in tqdm(range(100), desc='hold'):
    gripper.step_apply_force(dt, envs_idx=[0])
    hsr.step_whole_body_trajectory_batched(dt, envs_idx=[0])
    scene.step()
    frames.append(cam.render()[0])

print('Total frames captured:', len(frames))

## 8. Play the rendered animation inline

In [ ]:
# dt=0.02 → 50 sim Hz. Substeps don't change the wall-clock rate, so fps=50.
media.show_video(frames, fps=50)

## 9. (Optional) Save the video as an mp4

In [ ]:
output_path = '/content/ik_grasp_hsr.mp4'
media.write_video(output_path, frames, fps=50)
print('Saved video to', output_path)

# Download to your local machine if desired:
# try:
#     from google.colab import files
#     files.download(output_path)
# except Exception as e:
#     print('Not running in Colab or download blocked:', e)